# AgentEligibilityPolicy -- Circuit Breaker Integration

This notebook demonstrates how to use `AgentEligibilityPolicy` to automatically
remove failing agents from GroupChat speaker selection.

**Scenario (@marklysze):** `cheap_planner` always returns `None`, causing the
GroupChat to stall. With CB integration, after N failures, `cheap_planner` is
excluded and `pricey_planner` takes over.

## What is AgentEligibilityPolicy?

`AgentEligibilityPolicy` is a `typing.Protocol` that lets you filter
candidates before the speaker selection method (auto/round_robin/random/manual) runs.
Multiple policies can be registered; all must return `True` (AND semantics).

In [ ]:
from __future__ import annotations

from autogen import ConversableAgent, GroupChat, SelectionContext

try:
    from veronica_core import CircuitBreaker

    HAS_VERONICA = True
except ImportError:
    HAS_VERONICA = False
    print("veronica-core not installed. Using simplified CB demo instead.")

## Simple Circuit Breaker (no external dependency)

We implement a minimal circuit breaker to demonstrate the pattern.
The same interface works with veronica-core when installed.

In [ ]:
class SimpleCircuitBreaker:
    """Minimal circuit breaker: tracks per-agent failures."""

    def __init__(self, threshold: int = 2) -> None:
        self.threshold = threshold
        self._failure_counts: dict[str, int] = {}
        self._open: set[str] = set()

    def record_failure(self, name: str) -> None:
        self._failure_counts[name] = self._failure_counts.get(name, 0) + 1
        if self._failure_counts[name] >= self.threshold:
            self._open.add(name)
            print(f"[CB] {name} tripped OPEN after {self.threshold} failures")

    def is_open(self, name: str) -> bool:
        return name in self._open


class CircuitBreakerEligibilityPolicy:
    """AgentEligibilityPolicy backed by a SimpleCircuitBreaker."""

    def __init__(self, cb: SimpleCircuitBreaker) -> None:
        self.cb = cb

    def is_eligible(self, agent: ConversableAgent, ctx: SelectionContext) -> bool:
        eligible = not self.cb.is_open(agent.name)
        if not eligible:
            print(f"[Policy] {agent.name} excluded (circuit open)")
        return eligible

## Scenario: cheap_planner fails repeatedly

`cheap_planner` always returns `None` (simulates a broken LLM endpoint).
After 2 failures are recorded, the circuit opens and `cheap_planner` is excluded.
`pricey_planner` then becomes the only eligible candidate.

In [ ]:
cheap_planner = ConversableAgent(
    name="cheap_planner",
    llm_config=False,
    human_input_mode="NEVER",
    default_auto_reply=None,  # simulates failure
)

pricey_planner = ConversableAgent(
    name="pricey_planner",
    llm_config=False,
    human_input_mode="NEVER",
    default_auto_reply="I have a plan! Let us proceed.",
)

cb = SimpleCircuitBreaker(threshold=2)

# Simulate 2 failures -> circuit opens
cb.record_failure("cheap_planner")
cb.record_failure("cheap_planner")

policy = CircuitBreakerEligibilityPolicy(cb)

gc = GroupChat(
    agents=[cheap_planner, pricey_planner],
    messages=[],
    max_round=5,
    eligibility_policies=[policy],
)

# _prepare_and_select_agents is internal to GroupChat. We call it here to
# show the filtering in isolation without requiring a live LLM endpoint.
# In production code, pass eligibility_policies to GroupChat and run the
# chat normally via GroupChatManager -- policies are applied automatically
# during each speaker selection round.
selected, candidates, _ = gc._prepare_and_select_agents(pricey_planner)
print(f"Eligible candidates: {[a.name for a in candidates]}")
# Expected: ['pricey_planner']
print(f"Selected: {selected.name if selected else None}")

## External CB Integration (optional)

The `AgentEligibilityPolicy` protocol works with any circuit breaker
implementation. Below is an example using
[veronica-core](https://github.com/amabito/veronica-core), which provides
a distributed CB with Redis backend.

```
pip install "veronica-core[redis]>=1.8"
```

In [ ]:
if HAS_VERONICA:
    from veronica_core import CircuitBreaker, ExecutionContext

    class VeronicaEligibilityPolicy:
        """AgentEligibilityPolicy backed by veronica-core CircuitBreaker."""

        def __init__(self) -> None:
            self._breakers: dict[str, CircuitBreaker] = {}

        def get_breaker(self, name: str) -> CircuitBreaker:
            if name not in self._breakers:
                self._breakers[name] = CircuitBreaker(threshold=3, timeout=60.0)
            return self._breakers[name]

        def is_eligible(self, agent: ConversableAgent, ctx: SelectionContext) -> bool:
            breaker = self.get_breaker(agent.name)
            exec_ctx = ExecutionContext()
            result = breaker.check(exec_ctx)
            return result.allowed

    print("veronica-core VeronicaEligibilityPolicy ready")
else:
    print("veronica-core not available; using SimpleCircuitBreaker from above")